<a href="https://colab.research.google.com/github/oscarabrilarranz/Pruebas/blob/main/Top5IBEX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import pandas as pd
import requests
from io import StringIO


def leer_tabla_wikipedia(url):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/125.0 Safari/537.36"
        )
    }

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    return pd.read_html(StringIO(response.text))


def obtener_ibex35():
    url = "https://en.wikipedia.org/wiki/IBEX_35"
    tablas = leer_tabla_wikipedia(url)

    for tabla in tablas:
        cols = [str(c) for c in tabla.columns]

        if "Ticker" in cols:
            tickers = tabla["Ticker"].tolist()

            if "Company" in cols:
                nombres = tabla["Company"].tolist()
            elif "Name" in cols:
                nombres = tabla["Name"].tolist()
            else:
                nombres = tickers

            tickers = [
                str(t).strip() if str(t).strip().endswith(".MC")
                else str(t).strip() + ".MC"
                for t in tickers
            ]

            empresas = dict(zip(tickers, nombres))

            print(f"IBEX35 cargado: {len(tickers)} acciones")
            return tickers, empresas

    raise Exception("No se encontró la tabla del IBEX 35")


def obtener_datos_fundamentales(ticker):
    try:
        info = yf.Ticker(ticker).get_info()

        per = info.get("trailingPE")
        if per is None:
            per = info.get("forwardPE")

        deuda_total = info.get("totalDebt")
        equity = info.get("totalStockholderEquity")
        deuda_ratio = None

        if deuda_total is not None and equity not in (None, 0):
            deuda_ratio = deuda_total / equity

        beneficio = info.get("netIncomeToCommon")
        if beneficio is None:
            beneficio = info.get("netIncome")

        return {
            "PER": round(per, 2) if per is not None else None,
            "Ratio deuda": round(deuda_ratio, 2) if deuda_ratio is not None else None,
            "Beneficio neto": beneficio
        }

    except Exception:
        return {
            "PER": None,
            "Ratio deuda": None,
            "Beneficio neto": None
        }


def analizar_ibex35(tickers, empresas):
    print("\nDescargando precios del IBEX 35...")

    datos = yf.download(
        tickers,
        period="2d",
        interval="1d",
        auto_adjust=True,
        group_by="ticker",
        progress=False,
        threads=True
    )

    resultados = []

    for ticker in tickers:
        try:
            cierre_ayer = datos[ticker]["Close"].iloc[-2]
            cierre_hoy = datos[ticker]["Close"].iloc[-1]

            variacion = ((cierre_hoy - cierre_ayer) / cierre_ayer) * 100

            resultados.append({
                "Empresa": empresas.get(ticker, ""),
                "Ticker": ticker,
                "Cierre ayer": round(cierre_ayer, 2),
                "Cierre hoy": round(cierre_hoy, 2),
                "Variación %": round(variacion, 2)
            })

        except Exception:
            continue

    df = pd.DataFrame(resultados)

    if df.empty:
        return df

    print("Seleccionando las 5 mayores caídas y las 5 mayores subidas...")

    top_caidas = df.sort_values(by="Variación %", ascending=True).head(5)
    top_subidas = df.sort_values(by="Variación %", ascending=False).head(5)

    resultado = pd.concat([top_caidas, top_subidas], ignore_index=True)

    resultado["Tipo"] = ["Mayor caída"] * len(top_caidas) + ["Mayor subida"] * len(top_subidas)

    print("Obteniendo PER, ratio de deuda y beneficio neto...")

    fundamentales = resultado["Ticker"].apply(obtener_datos_fundamentales)
    fundamentales_df = pd.DataFrame(fundamentales.tolist())

    resultado = pd.concat(
        [resultado.reset_index(drop=True), fundamentales_df],
        axis=1
    )

    columnas = [
        "Tipo",
        "Empresa",
        "Ticker",
        "Cierre ayer",
        "Cierre hoy",
        "Variación %",
        "PER",
        "Ratio deuda",
        "Beneficio neto"
    ]

    return resultado[columnas]


def main():
    tickers, empresas = obtener_ibex35()

    resultado = analizar_ibex35(tickers, empresas)

    print("\n==============================")
    print("IBEX 35 - TOP 5 CAÍDAS Y SUBIDAS")
    print("==============================\n")

    if resultado.empty:
        print("No se pudieron obtener datos.")
        return

    print(resultado.to_string(index=False))

    nombre_csv = "ibex35_top5_caidas_subidas.csv"
    resultado.to_csv(nombre_csv, index=False)

    print(f"\nCSV generado: {nombre_csv}")


if __name__ == "__main__":
    main()

IBEX35 cargado: 35 acciones

Descargando precios del IBEX 35...
Seleccionando las 5 mayores caídas y las 5 mayores subidas...
Obteniendo PER, ratio de deuda y beneficio neto...

IBEX 35 - TOP 5 CAÍDAS Y SUBIDAS

        Tipo                      Empresa  Ticker  Cierre ayer  Cierre hoy  Variación %     PER Ratio deuda  Beneficio neto
 Mayor caída                    Ferrovial  FER.MC        58.06       57.30        -1.31   48.97        None       868000000
 Mayor caída                          ACS  ACS.MC       133.20      131.70        -1.13   35.79        None       991340032
 Mayor caída               Banco Sabadell  SAB.MC         3.27        3.24        -0.77   14.09        None      1104691968
 Mayor caída       Laboratorios Rovi [es] ROVI.MC        58.95       58.60        -0.59   24.83        None       140442000
 Mayor caída                        Sacyr SCYR.MC         4.61        4.59        -0.43   41.73        None        96529000
Mayor subida                      Solaria  S